In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

PROCESSED_DATA_DIR = Path("../processed_data")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_DIR = Path("../clean_data")
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
cols = ["Country", "VFN", "Mh", "T", "Va", "Ve", "Mk", "Cn", "m (kg)", "Ewltp (g/km)", "Ft", "Fm", "ec (cm3)", "ep (KW)", "z (Wh/km)", "year"]
csv_name = "4_eea_co2_emissions_from_passenger_cars-001.csv"
path = f"../data/{csv_name}"
df = spark.read.csv(path, header=True, inferSchema=True).select(*cols)

eu27_2020 = ['BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE']

unique_countries = df.select("Country").distinct()
df_only_eu = df[df.Country.isin(eu27_2020)]
df_only_eu.write.mode("overwrite").option("compression", "gzip").parquet(f"{CLEAN_DATA_DIR}/4_eea_co2_emissions_from_passenger_cars-001.parquet")

df.write.mode("overwrite").option("compression", "gzip").parquet(f"{PROCESSED_DATA_DIR}/4_eea_co2_emissions_from_passenger_cars-001.parquet")